# Checkpoint Evaluation

Compare trained checkpoints on the same fixed coloring and action-sampling seeds. This notebook performs no training and makes no archive writes.

In [2]:
# Imports and Project Paths

from pathlib import Path
from statistics import mean, median
from time import perf_counter

from ramsey import (
    RArchiveConstruction,
    REnvironment,
    REnvironmentConfig,
    RGraph,
    RMonochromaticObjective,
    RProblem,
    RRandomConstruction,
    RSQLiteArchive,
    RTabuMemory,
    RTabuMemoryConfig,
)
from ramsey.nn import (
    RCheckpointEvaluationConfig,
    RCheckpointEvaluator,
    build_evaluation_seeds,
    create_numpy_generator,
    resolve_torch_device,
)

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root "
        "or notebooks directory."
    )

In [3]:
# Evaluation Configuration

N_VERTICES = 43
ROLLOUT_STEPS = 128
EDGE_TABU_TENURE = 20
VISITED_STATE_WINDOW = 2_000

ARCHIVE_SCORE_LIMIT = 799
ARCHIVE_EVALUATION_SEEDS = 100
RANDOM_EVALUATION_SEEDS = 25
REPETITIONS_PER_SEED = 1
GREEDY = False

EVALUATION_RANDOM_SEED = 202_608_050
ACTION_RANDOM_SEED = 30_000

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

CHECKPOINT_DIRECTORY = (
    project_root
    / "checkpoints"
    / "phase_2"
)

CHECKPOINT_PATHS = (
    CHECKPOINT_DIRECTORY
    / "ramsey_policy_iteration_000199.pt",
    CHECKPOINT_DIRECTORY
    / "ramsey_policy_iteration_000249.pt",
)

In [4]:
# Runtime, Graph, and Archive

device = resolve_torch_device()
evaluation_rng = create_numpy_generator(
    EVALUATION_RANDOM_SEED
)

problem = RProblem.r55(
    n_vertices=N_VERTICES,
)

graph = RGraph(
    problem
)

existing_archive = globals().get("archive")

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(
    DATABASE_PATH
)

for checkpoint_path in CHECKPOINT_PATHS:
    if not checkpoint_path.is_file():
        archive.close()
        raise FileNotFoundError(checkpoint_path)

print("Project root:", project_root)
print("Device:", device)
print("Database:", DATABASE_PATH)
print("Database best:", archive.best_score(graph))

Project root: C:\code\RamseyNumber
Device: cuda:0
Database: C:\code\RamseyNumber\data\ramsey_colorings.sqlite3
Database best: 494


In [5]:
# Build the Fixed Evaluation Seed Set

archive_construction = RArchiveConstruction(
    archive=archive,
    rng=evaluation_rng,
    maximum_score=ARCHIVE_SCORE_LIMIT,
)

random_construction = RRandomConstruction(
    rng=evaluation_rng,
)

evaluation_seeds = (
    build_evaluation_seeds(
        graph,
        archive_construction,
        ARCHIVE_EVALUATION_SEEDS,
        name_prefix="archive",
    )
    + build_evaluation_seeds(
        graph,
        random_construction,
        RANDOM_EVALUATION_SEEDS,
        name_prefix="random",
    )
)

print("Fixed evaluation seeds:", len(evaluation_seeds))
print("Archive seeds:", ARCHIVE_EVALUATION_SEEDS)
print("Random seeds:", RANDOM_EVALUATION_SEEDS)

Fixed evaluation seeds: 125
Archive seeds: 100
Random seeds: 25


In [6]:
# Dedicated Evaluation Environment

evaluation_environment = REnvironment(
    graph=graph,
    objective=RMonochromaticObjective(),
    memory=RTabuMemory(
        number_of_edges=graph.number_of_edges,
        config=RTabuMemoryConfig(
            edge_tenure=EDGE_TABU_TENURE,
            visited_state_window=VISITED_STATE_WINDOW,
        ),
    ),
    config=REnvironmentConfig(
        max_steps=ROLLOUT_STEPS,
        use_aspiration=True,
    ),
)

checkpoint_evaluator = RCheckpointEvaluator(
    graph=graph,
    environment=evaluation_environment,
    device=device,
)

In [ ]:
# Evaluate Without Training

evaluation_start = perf_counter()

evaluation_result = checkpoint_evaluator.evaluate(
    checkpoint_paths=CHECKPOINT_PATHS,
    seeds=evaluation_seeds,
    config=RCheckpointEvaluationConfig(
        repetitions_per_seed=REPETITIONS_PER_SEED,
        action_seed=ACTION_RANDOM_SEED,
        greedy=GREEDY,
        score_thresholds=(
            600,
            550,
            500,
        ),
    ),
)

evaluation_elapsed = (
    perf_counter()
    - evaluation_start
)

print(
    "Evaluation time:",
    f"{evaluation_elapsed:.3f} seconds",
)

In [ ]:
# Aggregate Checkpoint Report

for evaluation in evaluation_result.evaluations:
    print()
    print("Checkpoint:", evaluation.checkpoint_path.name)
    print("Iteration:", evaluation.completed_iteration)
    print("Runs:", evaluation.number_of_runs)
    print(
        "Mean final score:",
        f"{evaluation.mean_final_score:.2f}",
    )
    print(
        "Median final score:",
        f"{evaluation.median_final_score:.2f}",
    )
    print(
        "Mean best score:",
        f"{evaluation.mean_best_score:.2f}",
    )
    print(
        "Median best score:",
        f"{evaluation.median_best_score:.2f}",
    )
    print(
        "Minimum best score:",
        evaluation.minimum_best_score,
    )
    print(
        "Mean best reduction:",
        f"{evaluation.mean_best_score_reduction:.2f}",
    )
    print(
        "Improved seeds:",
        f"{evaluation.improved_best_count}"
        f"/{evaluation.number_of_runs}",
    )
    print(
        "Threshold counts:",
        evaluation.threshold_counts,
    )

strongest = evaluation_result.strongest_checkpoint

print()
print(
    "Strongest checkpoint:",
    strongest.checkpoint_path.name,
)

In [ ]:
# Performance by Seed Source

for evaluation in evaluation_result.evaluations:
    print()
    print(evaluation.checkpoint_path.name)

    for source_name in evaluation.source_names:
        source_runs = evaluation.runs_for_source(
            source_name
        )

        print(
            f"  {source_name:24s}",
            f"runs={len(source_runs):3d}",
            f"mean-best="
            f"{mean(run.best_score for run in source_runs):.2f}",
            f"median-best="
            f"{median(run.best_score for run in source_runs):.2f}",
            f"minimum="
            f"{min(run.best_score for run in source_runs)}",
        )

In [ ]:
# Release the SQLite Connection

archive.close()
print("Archive closed.")